In [ ]:
import os

# 1. 确保在正确的文件夹
repo_name = "Diffusion-Illusions"
if not os.path.exists(repo_name):
    !git clone https://github.com/RyannDaGreat/Diffusion-Illusions
    %cd {repo_name}
    !pip install -r requirements.txt
    !pip install mediapy easydict "numpy<2.0"
else:
    if not os.getcwd().endswith(repo_name):
        %cd {repo_name}

print(f"当前工作目录: {os.getcwd()}")

In [ ]:
import os
import sys

# 1. 定义仓库文件夹名字
repo_name = "Diffusion-Illusions"

# 2. 检查文件夹是否存在，不存在则下载
if not os.path.exists(repo_name):
    print("⚠️ 未检测到项目文件夹，正在重新下载...")
    # 强制清理旧文件
    !rm -rf master.zip
    # 使用 ZIP 下载，避开 git clone 的网络问题
    !wget https://github.com/RyannDaGreat/Diffusion-Illusions/archive/refs/heads/master.zip -O master.zip
    !unzip -q master.zip
    # 重命名解压出来的文件夹
    !mv Diffusion-Illusions-master {repo_name}
    print("✅ 下载并解压完成")

# 3. 强制切换工作目录 (最关键的一步)
try:
    os.chdir(repo_name)
    print(f"✅ 已切换工作目录到: {os.getcwd()}")
except Exception as e:
    print(f"❌ 切换目录失败: {e}")

# 4. 强制添加路径到 Python 搜索列表 (双重保险)
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# 5. 补装可能缺失的依赖
print("正在检查依赖环境...")
!pip install mediapy easydict "numpy<2.0" > /dev/null 2>&1

print("\n🎉 环境修复完毕！现在请重新运行你的 import 代码块，应该不会报错了。")

In [ ]:
from rp import *
import numpy as np
import rp
import torch
import torch.nn as nn
import source.stable_diffusion as sd
from source.learnable_textures import LearnableImageFourier
from source.stable_diffusion_labels import NegativeLabel
from itertools import chain
import torchvision.transforms.functional as TF
import math
from google.colab import files
from PIL import Image, ImageOps

# === 核心工具函数 ===

def create_circular_mask(size=256):
    """创建圆形遮罩"""
    Y, X = np.ogrid[:size, :size]
    center = size / 2 - 0.5
    dist_from_center = np.sqrt((X - center)**2 + (Y - center)**2)
    mask = dist_from_center <= center
    return torch.from_numpy(mask).float().to(device).unsqueeze(0)

def rotate_tensor(image, angle_deg):
    """旋转图片并保持圆形裁剪"""
    rotated = TF.rotate(image, angle_deg, interpolation=TF.InterpolationMode.BILINEAR)
    return rotated * CIRCULAR_MASK

def apply_composite(layer_a, layer_b):
    """两张图叠加"""
    brightness = 3.0
    combined = (layer_a * layer_b * brightness)
    return combined.clamp(0, 99).tanh() * CIRCULAR_MASK

In [ ]:
# 初始化 GPU 和 模型
if 'model_sd' not in dir():
    print("正在加载 Stable Diffusion...")
    model_name = "CompVis/stable-diffusion-v1-4"
    gpu = rp.select_torch_device()
    model_sd = sd.StableDiffusion(gpu, model_name)
    device = model_sd.device
    CIRCULAR_MASK = create_circular_mask(256)
    print("模型加载完毕！")
else:
    print("模型已就绪。")
    if 'CIRCULAR_MASK' not in dir():
        CIRCULAR_MASK = create_circular_mask(256)

In [ ]:
# === 🔧 修改点：第 3 块 (降低目标亮度以减少鬼影) ===
import numpy as np
import os

target_tensors = {}
target_names = ["PKU", "EECS", "SMS"]

print(">>> 接下来请依次上传 3 张图片 <<<")
print("📉 【去鬼影模式】：我们会自动把字变暗一点，这样原图就看不出来了！")

for name in target_names:
    print(f"\n📂 请上传代表 【{name}】 的图片:")
    uploaded = files.upload()

    if uploaded:
        filename = next(iter(uploaded))
        img = Image.open(filename).convert('RGB')
        img = ImageOps.fit(img, (256, 256), method=Image.Resampling.LANCZOS)
        t = TF.to_tensor(img).to(device)

        # === 关键修改 ===
        # 之前的强力模式是: t * 0.8 + 0.1 (字太亮了，导致原图有痕迹)
        # 现在改为: t * 0.5 + 0.15
        # 解释：把字的最亮值限制在 0.65 左右。
        # 虽然叠出来的字会稍微暗一点，但原图的鬼影会大幅消失。
        t = t * 0.5 + 0.15

        t = t * CIRCULAR_MASK
        target_tensors[name] = t
        print(f"✅ {name} 处理完毕 (亮度已抑制)")
        rp.display_image(rp.as_numpy_image(t))
        os.remove(filename)

# 重新生成 Batch
target_list = []
for name in ["PKU", "EECS", "SMS"]:
    target_list.append(target_tensors[name])
targets_batch = torch.stack(target_list).to(device)
print("目标数据更新完毕。")

In [ ]:
import torch.nn.functional as F
from itertools import chain
import math

# === 1. 定义关键参数 (补全缺失的 ANGLES) ===
ANGLES = [0, 120, 240]  # <--- 这里定义了角度，必须要有！
GUIDANCE_STRENGTH = 10000

# === 2. 视觉迷彩 Prompt (高频纹理风格) ===
# 基底图 (Base)：故障艺术/全息风格
prompt_base = "Holographic iridescent texture, full spectrum rainbow colors, prismatic light, glowing white and colorful crystal details, abstract art, digital glitch"

# 旋转层 (Decoder)：彩色玻璃/霓虹风格
prompt_decoder = "Colorful stained glass mosaic, vibrant red green and blue glass fragments, kaleidoscope pattern, intricate geometry, bright neon colors, sci-fi circuit board"

# 负面提示词
negative_prompt = "smooth, blur, low resolution, organic, soft lighting, empty space, plain background"

# === 3. A100 专用加速器：BatchRotator ===
class BatchRotator:
    def __init__(self, angles_deg, size=256, device='cuda'):
        self.num_targets = len(angles_deg)
        self.device = device
        self.size = size
        thetas = []
        for angle in angles_deg:
            theta = math.radians(-angle) 
            c, s = math.cos(theta), math.sin(theta)
            row1 = [c, -s, 0]
            row2 = [s,  c, 0]
            thetas.append([row1, row2])
        self.theta_batch = torch.tensor(thetas, dtype=torch.float32, device=device)
        self.grid = F.affine_grid(self.theta_batch, [self.num_targets, 3, size, size], align_corners=True)

    def rotate_batch(self, image_tensor):
        img_batch = image_tensor.unsqueeze(0).expand(self.num_targets, -1, -1, -1)
        return F.grid_sample(img_batch, self.grid, mode='bilinear', padding_mode='zeros', align_corners=True)

# 初始化加速器
# 这里的 device 引用全局变量，如果报错请确认前面 Step 2 是否运行过
rotator = BatchRotator(ANGLES, size=256, device=device)

# === 4. 构建目标数据 Batch (防止 target_list 报错) ===
target_list = []
target_names = ["PKU", "EECS", "SMS"]

# 从上一上代码块生成的 target_tensors 中提取数据
for name in target_names:
    if 'target_tensors' in globals() and name in target_tensors:
        target_list.append(target_tensors[name])
    else:
        print(f"⚠️ 警告: 缺少 {name}，使用全黑填充防止报错 (请确保你运行了 Step 3 上传图片)")
        target_list.append(torch.zeros_like(CIRCULAR_MASK).repeat(3,1,1))

targets_batch = torch.stack(target_list).to(device)
print(f"✅ 目标数据构建完成，Batch shape: {targets_batch.shape}")

# === 5. 初始化模型参数 ===
image_maker = lambda: LearnableImageFourier(height=256, width=256, hidden_dim=256, num_features=256).to(device)
raw_layer_base = image_maker()
raw_layer_decoder = image_maker()

get_layer_base = lambda: raw_layer_base() * CIRCULAR_MASK
get_layer_decoder = lambda: raw_layer_decoder() * CIRCULAR_MASK

label_base = NegativeLabel(prompt_base, negative_prompt)
label_decoder = NegativeLabel(prompt_decoder, negative_prompt)

# 学习率
params = chain(raw_layer_base.parameters(), raw_layer_decoder.parameters())
optim = torch.optim.SGD(params, lr=1e-3)

print(f"✅ 初始化全部完成！Prompt风格: 高频纹理, 强度: {GUIDANCE_STRENGTH}")

In [ ]:
NUM_ITER = 3000
DISPLAY_INTERVAL = 200

model_sd.max_step = 980
model_sd.min_step = 20

display_eta = rp.eta(NUM_ITER, title='Training Status')

print(f"🚀 开始强力训练 (Strength={GUIDANCE_STRENGTH})...")

try:
    for iter_num in range(NUM_ITER):
        display_eta(iter_num)

        # --- A. 正常的 SD 训练 ---
        if iter_num % 2 == 0:
            curr_image = get_layer_decoder()
            curr_label = label_decoder
        else:
            curr_image = get_layer_base()
            curr_label = label_base

        # === 🔧 修改点 3：降低 SD 的控制力 ===
        # guidance_scale 从 80 降到 30
        # 让 AI "随意一点"，不要把纹理画得太死，方便我们藏东西
        # 在 s.train_step 里修改
        _ = model_sd.train_step(
            curr_label.embedding,
            curr_image[None],
            noise_coef=0.1,
            # 从 30 提回 50 或 60
            # 越高，原图越好看（鬼影越少），但藏的字可能会变淡一点
            guidance_scale=50
        )

        # --- B. 并行隐写引导 ---
        img_base = get_layer_base()
        img_decoder = get_layer_decoder()

        # 1. 旋转
        rotated_batch = rotator.rotate_batch(img_decoder)

        # 2. 合成
        base_batch = img_base.unsqueeze(0).expand(3, -1, -1, -1)
        mask_batch = CIRCULAR_MASK.unsqueeze(0)
        brightness = 3.0
        composite_batch = (base_batch * rotated_batch * brightness).clamp(0, 99).tanh() * mask_batch

        # 3. 计算 Loss (高权重)
        loss_secret = torch.mean((composite_batch - targets_batch)**2) * GUIDANCE_STRENGTH

        loss_secret.backward()

        # --- C. 显示进度 ---
        with torch.no_grad():
            if iter_num % DISPLAY_INTERVAL == 0:
                from IPython.display import clear_output
                clear_output(wait=True)

                base_np = rp.as_numpy_image(img_base)
                dec_np = rp.as_numpy_image(img_decoder)

                res_pku = rp.as_numpy_image(composite_batch[0])
                res_eecs = rp.as_numpy_image(composite_batch[1])
                res_sms = rp.as_numpy_image(composite_batch[2])

                blank = np.zeros_like(base_np)
                row1 = np.hstack([base_np, dec_np, blank])
                row2 = np.hstack([res_pku, res_eecs, res_sms])
                full_grid = np.vstack([row1, row2])

                print(f"Iteration {iter_num} / {NUM_ITER}")
                print(f"Weight: {GUIDANCE_STRENGTH} | Scale: 30")
                rp.display_image(full_grid)

        optim.step()
        optim.zero_grad()

except KeyboardInterrupt:
    print("训练停止。")